# MRI Diffusion Reconstruction Demo

End-to-end demo: synthetic k-space → retrospective undersampling → zero-filled reconstruction → diffusion reconstruction → SSIM/PSNR comparison.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print('mri-diffusion-recon demo')

## 1. Create Synthetic K-space Data

In [ ]:
def create_synthetic_mri(size=256):
    """create a synthetic MRI-like phantom (Shepp-Logan style)"""
    x = np.zeros((size, size))
    cx, cy = size // 2, size // 2
    Y, X = np.ogrid[:size, :size]

    # outer ellipse (skull)
    mask = ((X - cx) / (0.45 * size))**2 + ((Y - cy) / (0.55 * size))**2 <= 1
    x[mask] = 1.0

    # brain tissue regions
    for (cx_, cy_, rx, ry, val) in [
        (cx, cy, 0.3 * size, 0.38 * size, 0.8),
        (cx - 30, cy - 20, 0.12 * size, 0.10 * size, 0.4),
        (cx + 30, cy - 20, 0.12 * size, 0.10 * size, 0.4),
        (cx, cy + 25, 0.08 * size, 0.06 * size, 0.6),
    ]:
        region = ((X - cx_) / rx)**2 + ((Y - cy_) / ry)**2 <= 1
        x[region] = val

    return x


# create ground truth image
gt_image = create_synthetic_mri(size=256)

# compute 2D k-space via FFT
kspace_full = np.fft.fftshift(np.fft.fft2(gt_image))

print(f'ground truth image shape: {gt_image.shape}')
print(f'k-space shape: {kspace_full.shape}')
print(f'k-space dtype: {kspace_full.dtype}')
print(f'k-space magnitude range: [{np.abs(kspace_full).min():.2f}, {np.abs(kspace_full).max():.2f}]')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(gt_image, cmap='gray')
axes[0].set_title('ground truth MRI phantom')
axes[0].axis('off')

# log magnitude for visualization
kspace_viz = np.log1p(np.abs(kspace_full))
axes[1].imshow(kspace_viz, cmap='inferno')
axes[1].set_title('k-space magnitude (log scale)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 2. Retrospective Undersampling (4x Random Mask)

In [ ]:
def create_random_undersample_mask(shape, acceleration=4, center_fraction=0.08):
    """random undersampling mask with fully-sampled center region (4x acceleration)"""
    H, W = shape
    mask = np.zeros((H, W), dtype=np.float32)

    # always keep center low-frequency lines
    center_lines = int(W * center_fraction)
    c = W // 2
    mask[:, c - center_lines // 2: c + center_lines // 2] = 1.0

    # randomly sample remaining lines
    remaining = W - center_lines
    n_sample = int(W / acceleration) - center_lines
    n_sample = max(n_sample, 0)

    available_cols = [i for i in range(W) if mask[0, i] == 0]
    chosen = np.random.choice(available_cols, size=min(n_sample, len(available_cols)), replace=False)
    mask[:, chosen] = 1.0

    actual_acc = W / mask[0].sum()
    return mask, actual_acc


mask, actual_acc = create_random_undersample_mask((256, 256), acceleration=4)
print(f'target acceleration: 4x')
print(f'actual acceleration: {actual_acc:.2f}x')
print(f'sampling ratio: {mask.mean():.1%}')

# apply mask to k-space
kspace_undersampled = kspace_full * mask

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(mask, cmap='gray', aspect='auto')
axes[0].set_title(f'random undersampling mask (R={actual_acc:.1f}x)')
axes[0].set_xlabel('kx'); axes[0].set_ylabel('ky')

axes[1].imshow(np.log1p(np.abs(kspace_full)), cmap='inferno')
axes[1].set_title('full k-space')
axes[1].axis('off')

axes[2].imshow(np.log1p(np.abs(kspace_undersampled)), cmap='inferno')
axes[2].set_title('undersampled k-space')
axes[2].axis('off')
plt.tight_layout()
plt.show()

## 3. Zero-Filled Reconstruction

In [ ]:
# zero-filled: just inverse FFT the undersampled k-space
zf_recon = np.abs(np.fft.ifft2(np.fft.ifftshift(kspace_undersampled)))
zf_recon = zf_recon / zf_recon.max()

print(f'zero-filled reconstruction shape: {zf_recon.shape}')
print(f'aliasing artifacts from undersampling are visible (coherent noise due to random mask)')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(gt_image, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('ground truth')
axes[0].axis('off')

axes[1].imshow(zf_recon, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('zero-filled recon')
axes[1].axis('off')

diff = np.abs(gt_image - zf_recon)
axes[2].imshow(diff, cmap='hot', vmin=0, vmax=0.5)
axes[2].set_title('error map |gt - zf|')
axes[2].axis('off')
plt.tight_layout()
plt.show()

print(f'zero-filled MAE: {diff.mean():.4f}')

## 4. Diffusion Reconstruction (Simplified Forward Pass Demo)

In [ ]:
class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.SiLU(),
        )

    def forward(self, x):
        return self.conv(x)


class DiffusionReconUNet(nn.Module):
    """U-Net score network for MRI reconstruction"""
    def __init__(self, in_ch=2, base_ch=32):
        super().__init__()
        self.enc1 = UNetBlock(in_ch + 1, base_ch)  # +1 for time embedding
        self.enc2 = UNetBlock(base_ch, base_ch * 2)
        self.pool = nn.AvgPool2d(2)
        self.bottleneck = UNetBlock(base_ch * 2, base_ch * 4)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec2 = UNetBlock(base_ch * 4 + base_ch * 2, base_ch * 2)
        self.dec1 = UNetBlock(base_ch * 2 + base_ch, base_ch)
        self.out_conv = nn.Conv2d(base_ch, 1, 1)
        self.time_embed = nn.Embedding(1000, 1)  # simple time conditioning

    def forward(self, x_noisy, t, kspace_cond):
        B, _, H, W = x_noisy.shape
        t_emb = self.time_embed(t).view(B, 1, 1, 1).expand(B, 1, H, W)
        x_in = torch.cat([x_noisy, kspace_cond, t_emb], dim=1)
        e1 = self.enc1(x_in)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.dec2(torch.cat([self.up(b), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up(d2), e1], dim=1))
        return self.out_conv(d1)


def diffusion_reconstruct(zf_img, kspace_us, mask, model, T=50, device='cpu'):
    """simplified DDPM reverse process with data consistency projection"""
    model.eval()

    # prepare tensors
    x = torch.tensor(zf_img, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    kspace_cond_real = torch.tensor(np.real(kspace_us) / (np.abs(kspace_full).max() + 1e-8),
                                     dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    mask_t = torch.tensor(mask, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

    x_t = x + 0.3 * torch.randn_like(x)  # start from noisy zero-filled

    history = [x_t[0, 0].detach().cpu().numpy()]

    with torch.no_grad():
        for t_step in range(T - 1, -1, -1):
            t_tensor = torch.full((1,), t_step, dtype=torch.long, device=device)
            noise_pred = model(x_t, t_tensor, kspace_cond_real)
            sigma = t_step / T
            x_t = x_t - sigma * 0.05 * noise_pred
            # data consistency: replace measured k-space lines
            if t_step % 10 == 0:
                x_np = x_t[0, 0].cpu().numpy()
                kspace_pred = np.fft.fftshift(np.fft.fft2(x_np))
                kspace_dc = kspace_pred * (1 - mask) + kspace_us  # keep measured lines
                x_dc = np.abs(np.fft.ifft2(np.fft.ifftshift(kspace_dc)))
                x_dc = x_dc / (x_dc.max() + 1e-8)
                x_t = torch.tensor(x_dc, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
                history.append(x_dc)

    return x_t[0, 0].cpu().numpy(), history


model = DiffusionReconUNet(in_ch=2).eval()
print(f'diffusion U-Net params: {sum(p.numel() for p in model.parameters()):,}')

diffusion_recon, history = diffusion_reconstruct(zf_recon, kspace_undersampled, mask, model)
diffusion_recon = np.clip(diffusion_recon, 0, 1)

# show reconstruction progression
fig, axes = plt.subplots(1, min(5, len(history)), figsize=(14, 3))
for i, (ax, img) in enumerate(zip(axes, history[:5])):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'step {i * 10}')
    ax.axis('off')
plt.suptitle('diffusion reconstruction progression')
plt.tight_layout()
plt.show()
print(f'final reconstruction shape: {diffusion_recon.shape}')

## 5. SSIM / PSNR Comparison

In [ ]:
def psnr(gt, pred, data_range=1.0):
    mse = np.mean((gt - pred)**2)
    if mse == 0:
        return float('inf')
    return 20 * np.log10(data_range / np.sqrt(mse))


def ssim(gt, pred, data_range=1.0, window=11):
    """simplified SSIM computation"""
    C1 = (0.01 * data_range)**2
    C2 = (0.03 * data_range)**2
    mu_gt = np.mean(gt)
    mu_pred = np.mean(pred)
    sigma_gt = np.var(gt)
    sigma_pred = np.var(pred)
    sigma_cross = np.mean((gt - mu_gt) * (pred - mu_pred))
    num = (2 * mu_gt * mu_pred + C1) * (2 * sigma_cross + C2)
    den = (mu_gt**2 + mu_pred**2 + C1) * (sigma_gt + sigma_pred + C2)
    return num / den


gt_norm = gt_image / gt_image.max()

metrics = {
    'Zero-Filled': {'psnr': psnr(gt_norm, zf_recon), 'ssim': ssim(gt_norm, zf_recon)},
    'Diffusion Recon': {'psnr': psnr(gt_norm, diffusion_recon), 'ssim': ssim(gt_norm, diffusion_recon)},
}

print('reconstruction quality metrics')
print('=' * 45)
print(f'{"Method":<20} {"PSNR (dB)":>12} {"SSIM":>10}')
print('-' * 45)
for method, m in metrics.items():
    print(f'{method:<20} {m["psnr"]:>12.2f} {m["ssim"]:>10.4f}')

# visual comparison
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
imgs = [gt_norm, zf_recon, diffusion_recon]
titles = ['ground truth', 'zero-filled', 'diffusion recon']

for i, (ax, img, title) in enumerate(zip(axes[0], imgs, titles)):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')

# error maps
axes[1][0].axis('off')
axes[1][0].text(0.5, 0.5, 'error maps below', ha='center', va='center', fontsize=12)
for i, (ax, img, title) in enumerate(zip(axes[1][1:], [zf_recon, diffusion_recon], ['ZF error', 'Diffusion error'])):
    err = np.abs(gt_norm - img)
    im = ax.imshow(err, cmap='hot', vmin=0, vmax=0.3)
    ax.set_title(f'{title} (MAE={err.mean():.4f})')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('MRI reconstruction comparison (4x undersampling)', fontsize=12)
plt.tight_layout()
plt.show()